# CNN (ResNet-18) for Solar Panel Fault Detection

Compare with Vision Transformer (86%)

---

## 1. Import Libraries

In [14]:
import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Libraries imported
PyTorch: 2.9.1+cu128
CUDA available: True
GPU: Quadro RTX 6000


## 2. Configuration

In [15]:
TRAIN_PATH = "/home/compute.ashesi.lan/e.bilson/Fault_detect_V2/data_v2/images/test/"

TEST_PATH = "/home/compute.ashesi.lan/e.bilson/Fault_detect_V2/data_v2/images/train/"

CLASS_NAMES = [
    'Cell-Fault', 'Cracking', 'Diode-Fault', 'Hot-Spot',
    'No-Anomaly', 'Offline-Module', 'Shadowing', 'Soiling', 'Vegetation'
]

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 100
LEARNING_RATE = 0.0001
NUM_CLASSES = len(CLASS_NAMES)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

print("="*60)
print("CNN Configuration")
print("="*60)
print(f"Train: {TRAIN_PATH}")
print(f"Test: {TEST_PATH}")
print(f"Classes: {NUM_CLASSES}")
print(f"Device: {device}")

CNN Configuration
Train: /home/compute.ashesi.lan/e.bilson/Fault_detect_V2/data_v2/images/test/
Test: /home/compute.ashesi.lan/e.bilson/Fault_detect_V2/data_v2/images/train/
Classes: 9
Device: cuda


## 3. Data Augmentation

In [16]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.GaussianBlur(p=0.3),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2()
])

test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2()
])

print("Augmentation ready")

Augmentation ready


## 4. Dataset Class

In [17]:
class SolarDataset(Dataset):
    def __init__(self, root_path, class_names, transform=None):
        self.root_path = root_path
        self.class_names = class_names
        self.transform = transform
        self.samples = []
        
        for class_idx, class_name in enumerate(class_names):
            class_dir = os.path.join(root_path, class_name)
            if not os.path.exists(class_dir):
                continue
                
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.samples.append((os.path.join(class_dir, fname), class_idx))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        if self.transform:
            img = self.transform(image=img)['image']
        
        return img, label

print("Dataset class ready")

Dataset class ready


## 5. Load Data

In [18]:
print("="*60)
print("Loading Data")
print("="*60)

train_dataset = SolarDataset(TRAIN_PATH, CLASS_NAMES, train_transform)
test_dataset = SolarDataset(TEST_PATH, CLASS_NAMES, test_transform)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_subset, val_subset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size]
)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {train_size}")
print(f"Val: {val_size}")
print(f"Test: {len(test_dataset)}")
print("Data loaded!")

Loading Data
Train: 1608
Val: 403
Test: 15996
Data loaded!


## 6. Create CNN Model

In [19]:
model = timm.create_model('resnet18', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())

print("="*60)
print("CNN Model (ResNet-18)")
print("="*60)
print(f"Parameters: {total_params:,}")
print(f"Device: {device}")
print("Model ready")

CNN Model (ResNet-18)
Parameters: 11,181,129
Device: cuda
Model ready


## 7. Training Setup

In [20]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

print("Training setup complete")

Training setup complete


## 8. Training Functions

In [21]:
def train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100.0 * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100.0 * correct / total

print("Training functions ready")

Training functions ready


## 9. Train Model

In [22]:
print("="*60)
print("Training CNN")
print("="*60)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
patience = 10
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_cnn.pth')
        print(f"Saved! Best Val: {val_acc:.2f}%")
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stop at epoch {epoch+1}")
        break

print(f"\nTraining done! Best: {best_val_acc:.2f}%")
model.load_state_dict(torch.load('best_cnn.pth'))

Training CNN

Epoch 1/100


Validating: 100%|██████████| 13/13 [00:00<00:00, 17.38it/s]


Train Loss: 2.0332, Acc: 38.43%
Val Loss: 1.8686, Acc: 54.59%
Saved! Best Val: 54.59%

Epoch 2/100


Validating: 100%|██████████| 13/13 [00:00<00:00, 17.76it/s]


Train Loss: 1.7942, Acc: 48.51%
Val Loss: 1.7097, Acc: 54.59%

Epoch 3/100


...

Validating: 100%|██████████| 13/13 [00:00<00:00, 17.48it/s]


Train Loss: 1.4056, Acc: 62.44%
Val Loss: 1.3710, Acc: 65.26%

Epoch 32/100


Validating: 100%|██████████| 13/13 [00:00<00:00, 18.85it/s]

Train Loss: 1.3972, Acc: 62.19%
Val Loss: 1.3806, Acc: 64.27%
Early stop at epoch 32

Training done! Best: 65.76%


<All keys matched successfully>

## 10. Evaluate

In [ ]:
print("="*60)
print("Testing CNN")
print("="*60)

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = accuracy_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)

print(f"\nCNN Test Accuracy: {test_acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

## 11. Plot Training

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train', linewidth=2)
ax1.plot(history['val_loss'], label='Val', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('CNN Training Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='Train', linewidth=2)
ax2.plot(history['val_acc'], label='Val', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('CNN Training Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_training.png', dpi=150)
plt.show()
print("Saved: cnn_training.png")

## 12. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)

ax.set_title('CNN Confusion Matrix', fontsize=14, pad=20)
ax.set_ylabel('True')
ax.set_xlabel('Predicted')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('cnn_confusion.png', dpi=150)
plt.show()
print("Saved: cnn_confusion.png")

## 13. CNN vs Vision Transformer

In [ ]:
VIT_ACCURACY = 86.0

results = {
    'YOLOv8 v1.0': 77.4,
    'YOLOv8 v2.0': 77.0,
    'CNN (ResNet-18)': test_acc * 100,
    'Vision Transformer': VIT_ACCURACY,
    'Paper (ResNet)': 85.9
}

print("="*60)
print("COMPARISON")
print("="*60)
for name, acc in results.items():
    print(f"{name:25s}: {acc:6.2f}%")

winner = 'CNN' if test_acc*100 > VIT_ACCURACY else 'Vision Transformer'
diff = abs(test_acc*100 - VIT_ACCURACY)
print(f"\nWinner: {winner} (by {diff:.1f}%)")

fig, ax = plt.subplots(figsize=(12, 6))
methods = list(results.keys())
accs = list(results.values())
colors = ['skyblue', 'lightcoral', 'orange', 'lightgreen', 'gold']

bars = ax.bar(methods, accs, color=colors, alpha=0.8, edgecolor='black', linewidth=2)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('CNN vs Transformer Comparison', fontsize=14, fontweight='bold')
ax.set_ylim([0, 100])
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')

plt.tight_layout()
plt.savefig('comparison.png', dpi=150)
plt.show()
print("Saved: comparison.png")

## 14. Summary

In [ ]:
print("="*60)
print("FINAL RESULTS")
print("="*60)
print(f"\nCNN (ResNet-18):")
print(f"  Test Accuracy: {test_acc*100:.2f}%")
print(f"  Parameters: {total_params:,}")
print(f"  Epochs: {len(history['train_loss'])}")

print(f"\nVision Transformer:")
print(f"  Test Accuracy: {VIT_ACCURACY:.2f}%")

print(f"\nComparison:")
if test_acc*100 > VIT_ACCURACY:
    print(f"  CNN WINS by {test_acc*100 - VIT_ACCURACY:.1f}%")
    print("  Recommendation: Use CNN (faster + better)")
elif test_acc*100 < VIT_ACCURACY:
    print(f"  ViT WINS by {VIT_ACCURACY - test_acc*100:.1f}%")
    print("  Recommendation: Use ViT (best accuracy)")
else:
    print("  TIE! Both excellent!")
    print("  Recommendation: Use CNN (faster, smaller)")

print(f"\nFiles created:")
print("  - cnn_training.png")
print("  - cnn_confusion.png")
print("  - comparison.png")
print("  - best_cnn.pth")

print("\n" + "="*60)
print("CNN vs Transformer comparison complete!")
print("="*60)